In [21]:
import os
import sys
import time
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import warnings
warnings.filterwarnings('ignore')

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve, classification_report
from sklearn.datasets import make_classification
from xgboost import XGBClassifier
from xgboost import plot_importance
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

import lightgbm as lgbm

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from hyperopt import hp
import HyperParams as HP

import importlib
importlib.reload(HP)
importlib.reload(hs)

from utils import user_utils
from utils import preprocessing
from utils import data_sampling
from utils import modeling
from utils import hyperopt_search as hs

In [3]:
raw_df = pd.read_csv('../data/creditcard.csv')

In [4]:
X_features, y_target = preprocessing.split_features_target(raw_df, cols= 'Time')

In [5]:
cap_X_feature = preprocessing.cap_outliers(X_features)

In [6]:
X_train, X_test, y_train, y_test = preprocessing.data_split(cap_X_feature, y_target)
X_tr, X_val, y_tr, y_val = preprocessing.data_split(X_train, y_train, size=0.4)

In [7]:
# 결과받을 딕셔너리
results = {}

In [ ]:
tuner = user_utils.HyperOptTuner(max_evals=100, random_state=23)

catboost_search_space = {
        'iterations': hp.quniform('iterations', 100, 1000, 50),
        'depth': hp.quniform('depth', 4, 10, 1),  
        'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),  
        'l2_leaf_reg': hp.loguniform('l2_leaf_reg', np.log(1), np.log(10)),  
        'border_count': hp.choice('border_count', [32, 64, 128, 254]),  
        'bagging_temperature': hp.uniform('bagging_temperature', 0, 1),  
        'random_strength': hp.uniform('random_strength', 0, 10)  
    }
dt_search_space = {
        'criterion': hp.choice('criterion', ['gini', 'entropy']),
        'max_depth': hp.quniform('max_depth', 5, 20, 1),
        'min_samples_split': hp.quniform('min_samples_split', 2, 50, 2),
        'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 20, 1), 
        'max_features': hp.choice('max_features', ['sqrt', 'log2', None]),
        'min_impurity_decrease': hp.uniform('min_impurity_decrease', 0, 0.01)
    }

catboost = CatBoostClassifier()
dt_clf = DecisionTreeClassifier()

best_params, best_catboost, trials, exec_time = tuner.tune(
    catboost, X_tr, y_tr, X_val, y_val, catboost_search_space
    )

model_name = 'cb_ho_best'
results[model_name] = user_utils.get_model_train_eval(
    best_catboost, model_name, X_train, X_test, y_train, y_test, best_params
    )



CatBoostClassifier 튜닝 시작
100%|██████████| 100/100 [24:59<00:00, 15.00s/trial, best loss: -0.8037974683544303]

튜닝 시간: 1499.90초
최적 recall: 0.8038

최적 모델의 전체 평가 점수:
- roc_auc: 0.9828
- f1: 0.8467
- precision: 0.8944
- recall: 0.8038
- accuracy: 0.9995

최적 하이퍼파라미터:
-bagging_temperature: 0.5879957646583531
-border_count: 64
-depth: 4
-iterations: 550
-l2_leaf_reg: 7.548114015892266
-learning_rate: 0.017750712329165527
-random_strength: 1.722307004315998
-random_state: 23
-verbose: 0
-allow_writing_files: False
✓ 모델 저장 완료: ../models\cb_ho_best.pkl
  파일 크기: 0.18 MB
folder = c:\big20\git\big20-ML-project2-team3\CreditCardFraud\results
{'AUC': 0.9685, '정확도': 0.9996, '정밀도': 0.9412, '재현율': 0.8163, 'F1': 0.8743 }
{'오차행렬':
[[56859     5]
 [   18    80]] }
실행 시간: 14.242465257644653
하이퍼파라미터: {'bagging_temperature': 0.5879957646583531, 'border_count': 64, 'depth': 4, 'iterations': 550, 'l2_leaf_reg': 7.548114015892266, 'learning_rate': 0.017750712329165527, 'random_strength': 1.722307004315998, 'ran

In [ ]:
modeling.model_metrics_graph(results, 'cb모델 성능지표 비교')

In [14]:
best_params, best_dt, trials, exec_time = tuner.tune(
    dt_clf, X_tr, y_tr, X_val, y_val, dt_search_space
    )

model_name = 'dt_ho_best'
results[model_name] = user_utils.get_model_train_eval(
    best_dt, model_name, X_train, X_test, y_train, y_test, best_params
    )


DecisionTreeClassifier 튜닝 시작
100%|██████████| 100/100 [02:10<00:00,  1.30s/trial, best loss: -0.7911392405063291]

튜닝 시간: 130.03초
최적 recall: 0.7911

최적 모델의 전체 평가 점수:
- roc_auc: 0.9245
- f1: 0.7886
- precision: 0.7862
- recall: 0.7911
- accuracy: 0.9993

최적 하이퍼파라미터:
-criterion: entropy
-max_depth: 8
-max_features: None
-min_impurity_decrease: 0.0004705069764146597
-min_samples_leaf: 10
-min_samples_split: 18
-random_state: 23
✓ 모델 저장 완료: ../models\dt_ho_best.pkl
  파일 크기: 0.00 MB
folder = c:\big20\git\big20-ML-project2-team3\CreditCardFraud\results
{'AUC': 0.9361, '정확도': 0.9993, '정밀도': 0.7664, '재현율': 0.8367, 'F1': 0.8000 }
{'오차행렬':
[[56839    25]
 [   16    82]] }
실행 시간: 4.390155076980591
하이퍼파라미터: {'criterion': 'entropy', 'max_depth': 8, 'max_features': None, 'min_impurity_decrease': 0.0004705069764146597, 'min_samples_leaf': 10, 'min_samples_split': 18, 'random_state': 23}


In [ ]:

modeling.model_metrics_graph(results, 'dt모델 성능지표 비교')

In [40]:

X_train, X_test, y_train, y_test = preprocessing.data_split(cap_X_feature, y_target)

In [41]:

X_over, y_over = data_sampling.oversampling_smote(X_train, y_train)

✅ SMOTE 오버샘플링 완료
   원본 샘플 수: 227845 (Class 0: 227451, Class 1: 394)
   샘플링 후: 454902 (Class 0: 227451, Class 1: 227451)


In [42]:
# 5.2 Over Sampling한 경우 학습/검증 데이터 분리
X_tr_over, X_val_over, y_tr_over, y_val_over = preprocessing.data_split(X_over, y_over, size=0.4)

In [ ]:
# 2. Stacking Classifier (추천!)
from sklearn.ensemble import StackingClassifier

# Base models (다양성 확보)
base_models = [
    ('rf', RandomForestClassifier(
        random_state= 23,
        n_estimators= 390,
        max_depth= 25,
        class_weight= 'balanced',
        min_samples_leaf= 1,
        min_samples_split= 7,
        n_jobs= -1)
     ),
    ('xgb', XGBClassifier(
        colsample_bytree = 1.0,
        gamma = 1.7089112007254605,
        learning_rate = 0.038583724829444874,
        max_depth = 3,
        min_child_weight = 6,
        n_estimators = 150,
        reg_alpha = 0.5940048544595263,
        reg_lambda = 0.007186370173139192,
        scale_pos_weight = 82.0,
        subsample = 0.6,
        random_state = 23
        )
     ),
    ('lgbm', LGBMClassifier(
        random_state = 23,
        n_estimators = 400,
        num_leaves = 36,
        learning_rate = 0.03,
        subsample = 0.9,
        colsample_bytree = 0.75,
        reg_alpha = 0.6,
        reg_lambda = 0.2,
        class_weight ='balanced',
        n_jobs = -1,)
     ),
    ('gb', GradientBoostingClassifier(
        learning_rate = 0.02083271293464231, 
        max_depth = 3, 
        n_estimators = 800, 
        subsample = 0.7076395576782778, 
        random_state = 23)
     )
]

# Meta model
meta_model = LogisticRegression(class_weight='balanced')

stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    stack_method='predict_proba'
)

model_name = 'stacking_ho-best_smote'
results[model_name] = user_utils.get_model_train_eval(
    stacking_clf, model_name, X_train, X_test, y_train, y_test, best_params
    )

[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017224 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7395
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [3]:
mlp_search_space = {
    # hidden_layer_sizes: 은닉층 구조
    'hidden_layer_sizes': hp.choice('hidden_layer_sizes', [
        (50,),
        (100,),
        (50, 50),
        (100, 50),
        (100, 100),
        (100, 50, 25)
    ]),
    # activation: 활성화 함수
    'activation': hp.choice('activation', ['relu', 'tanh', 'logistic']),
    # solver: 최적화 알고리즘
    'solver': hp.choice('solver', ['adam', 'sgd']),
    # alpha: L2 정규화 파라미터
    'alpha': hp.loguniform('alpha', np.log(0.0001), np.log(0.1)),
    # learning_rate: 학습률 스케줄
    'learning_rate': hp.choice('learning_rate', ['constant', 'invscaling', 'adaptive']),
    # learning_rate_init: 초기 학습률
    'learning_rate_init': hp.loguniform('learning_rate_init', np.log(0.0001), np.log(0.01)),
    # batch_size: 미니배치 크기
    'batch_size': hp.choice('batch_size', [32, 64, 128, 'auto']),
    # max_iter: 최대 반복 횟수
    'max_iter': hp.choice('max_iter', [200, 500, 1000]),
    # early_stopping: 조기 종료
    'early_stopping': hp.choice('early_stopping', [True, False]),
    # validation_fraction: 검증 데이터 비율 (early_stopping=True일 때)
    'validation_fraction': 0.1,
    # n_iter_no_change: 개선 없을 때 종료까지 반복 횟수
    'n_iter_no_change': 10,
    # random_state: 재현성
    'random_state': 42
}

In [ ]:
search_result = hs.hyperopt_search(
        model_class=MLPClassifier,
        search_space=mlp_search_space,  # svm_rbf_search_space → mlp_search_space
        X_train=X_train,
        y_train=y_train,
        max_evals=100,  # MLP는 SVM보다 빠르므로 100회 권장
        cv=5,
        scoring='roc_auc',  # 또는 'f1', 'recall'
        random_state=23,
        save_trials=True,
        trials_path='../models/trials/',
        verbose=True,
        use_gpu=False  # sklearn MLPClassifier는 GPU 미지원
    )

  MLPClassifier 하이퍼파라미터 탐색 시작
 25%|██▌       | 25/100 [1:03:18<2:07:55, 102.35s/trial, best loss: -0.9872223843783823]

In [20]:
final_result = hs.train_and_evaluate(
        model_class=MLPClassifier,
        params=HP.mlp_best_param,
        X_train=X_train,  # 전체 학습 데이터 사용
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        save_model=True,
        save_model_path='../models/',
        save_result_path='../results/',
        verbose=True
    )

  MLPClassifier 최종 학습 및 평가
사용 파라미터: {'activation': 'tanh', 'alpha': 0.004721246220578279, 'batch_size': 3, 'early_stopping': True, 'hidden_layer_sizes': (100,), 'learning_rate': 'adaptive', 'learning_rate_init': 0.00036361698972348403, 'max_iter': 1, 'solver': 'adam', 'random_state': 23}
✓ 학습 완료 (15.98초)


NameError: name 'pred' is not defined